# SRL Extraction Demo with Sample Data

This notebook runs the Semantic Role Labeling (SRL) extraction pipeline on the small, self-contained subset in [`sample_data/`](sample_data/). It is the runnable companion to the full notebooks in [`notebooks/`](notebooks/).

**No OntoNotes 5.0 download or LDC license is required.**

The notebook:

1. reads matching PropBank `.prop` and OntoNotes `.onf` files;
2. aligns each proposition with its sentence and parse tree;
3. converts PropBank tree pointers into surface text spans;
4. removes trace anchors and builds the final dataset;
5. checks the result against `sample_data/expected_output.csv`.

The expected output has **107 rows** and five columns: `SENTENCE`, `PREDICATES`, `ARG0`, `ARG1`, and `Merged_Arguments`.


## 1. Locate the sample data

The notebook starts from the repository root and reads `sample_data/`. If it is opened directly in Google Colab, the repository is not present, so the next cell performs a sparse Git checkout that downloads only `sample_data/` instead of the large notebooks.


In [ ]:
from pathlib import Path
import os
import re
import subprocess

import pandas as pd
from nltk.corpus.reader.propbank import PropbankTreePointer
from nltk.tree import Tree

DATA_DIR = Path("sample_data")

# The notebook lives at the repository root. If it is launched from a
# subdirectory, look one level up before falling back to a Colab checkout.
if not DATA_DIR.exists() and (Path("..") / "sample_data").exists():
    DATA_DIR = Path("..") / "sample_data"

# On Colab the repository is not present. A sparse checkout downloads only
# sample_data/ instead of the large notebooks.
if not DATA_DIR.exists():
    repo = Path("srl-data-extraction")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--filter=blob:none",
                "--sparse",
                "https://github.com/Amirmohammadomidi/srl-data-extraction.git",
            ],
            check=True,
        )
    os.chdir(repo)
    subprocess.run(["git", "sparse-checkout", "set", "sample_data"], check=True)
    DATA_DIR = Path("sample_data")

print(f"Sample data directory: {DATA_DIR}")

In [ ]:
prop_files = sorted(DATA_DIR.glob("*.prop"))
sample_files = [
    (prop_path, prop_path.with_suffix(".onf"))
    for prop_path in prop_files
    if prop_path.with_suffix(".onf").exists()
]

print(f"Found {len(sample_files)} sample pairs:")
for prop_path, onf_path in sample_files:
    print(f"  {prop_path.name:16s} {onf_path.name}")

## 2. Parse the OntoNotes `.onf` files

Each `.onf` sentence block contains a plain sentence, a treebanked sentence, a parse tree, and a leaf/role listing. The pipeline needs the treebanked sentence (whose tokens match the parse tree leaves) and the tree itself. The plain sentence is extracted too, for inspection.


In [ ]:
SEPARATOR = "-" * 120

def read_onf(path):
    """Return plain sentences, treebanked sentences, and parse trees from an .onf file."""
    blocks = path.read_text(encoding="utf-8").split("\n\n")
    plain, treebanked, trees = [], [], []
    for i, block in enumerate(blocks):
        if block in (SEPARATOR, "\n" + SEPARATOR):
            plain.append(blocks[i + 1].split("\n---------------\n    ")[1].replace("\n   ", ""))
            treebanked.append(blocks[i + 2].split("\n--------------------\n    ")[1].replace("\n   ", ""))
            trees.append(blocks[i + 3].split("Tree:\n-----\n    (TOP ")[1][:-1])
    return plain, treebanked, trees

plain, treebanked, trees = read_onf(sample_files[0][1])
print(f"{sample_files[0][1].name}: {len(plain)} sentences")
print("Plain sentence:     ", plain[0])
print("Treebanked sentence:", treebanked[0])
print("Parse tree prefix:  ", trees[0][:80], "...")

## 3. Parse the PropBank `.prop` files

Each line of a `.prop` file is one predicate-argument proposition. The second field is the sentence index, and tokens such as `12:0-ARG0`, `14:2-ARG1`, and `13:0-rel` are tree pointers.


In [ ]:
def read_prop(path):
    """Return the non-empty proposition lines from a .prop file."""
    return [line for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

propositions = read_prop(sample_files[0][0])
print(f"{sample_files[0][0].name}: {len(propositions)} propositions")
print(propositions[0])

## 4. Align propositions with sentences and trees

The sentence index in each proposition selects the matching treebanked sentence and parse tree from the `.onf` file. Repeating this for every sample produces one row per predicate-argument proposition.


In [ ]:
annotations, treebanked_sentences, tree_strings = [], [], []
for prop_path, onf_path in sample_files:
    propositions = read_prop(prop_path)
    _, treebanked, trees = read_onf(onf_path)
    for proposition in propositions:
        sentence_index = int(proposition.split()[1])
        annotations.append(proposition)
        treebanked_sentences.append(treebanked[sentence_index])
        tree_strings.append(trees[sentence_index])

print(f"Aligned {len(annotations)} propositions.")

## 5. Extract role pointers

For each proposition, the pointers ending in `-ARG0`, `-ARG1`, and `-rel` are extracted. Compound pointers such as `0:2*6:1` represent discontinuous spans and are split into individual pointers.


In [ ]:
def extract_role(annotation, suffix):
    """Return the tree pointer for one role suffix, or '' when the role is absent."""
    pointer = ""
    for token in annotation.split():
        if token.endswith(suffix):
            pointer = token.split("-")[0]
    return pointer

def split_pointers(pointer_string):
    """Split compound PropBank pointers such as '0:2*6:1' into individual pointers."""
    for delimiter in ("*", ",", ";"):
        pointer_string = " ".join(pointer_string.split(delimiter))
    return pointer_string.split()

roles = {}
for role, suffix in (("ARG0", "-ARG0"), ("ARG1", "-ARG1"), ("REL", "-rel")):
    raw = [extract_role(annotation, suffix) for annotation in annotations]
    roles[role] = [split_pointers(value) if value else value for value in raw]

print("Proposition:   ", annotations[0])
print("ARG0 pointers: ", roles["ARG0"][0])
print("ARG1 pointers: ", roles["ARG1"][0])
print("REL pointer:   ", roles["REL"][0])

## 6. Convert pointers to text spans

NLTK's `PropbankTreePointer.select` returns the subtree addressed by a pointer; joining its leaves gives the surface text span used by the dataset.


In [ ]:
def pointer_to_span(pointer_string, tree_string):
    """Convert one PropBank pointer such as '0:1' to its surface text span."""
    parts = re.split(";|:", pointer_string)
    pointer = PropbankTreePointer(int(parts[0]), int(parts[1]))
    selected = pointer.select(Tree.fromstring(tree_string))
    return " ".join(selected.leaves())

spans = {}
for role, pointer_lists in roles.items():
    role_spans = []
    for pointers, tree_string in zip(pointer_lists, tree_strings):
        if not pointers:
            role_spans.append("")
        else:
            role_spans.append(" ".join(pointer_to_span(p, tree_string) for p in pointers))
    spans[role] = role_spans

print("REL: ", spans["REL"][:3])
print("ARG0:", spans["ARG0"][:3])
print("ARG1:", spans["ARG1"][:3])

## 7. Remove traces and build the dataset

The regex strips trace anchors such as `*T*-10`, `*PRO*-2`, `*?*`, and `*-1`, then normalizes whitespace. The pattern is intentionally identical to the full notebooks so this demo reproduces their output exactly. Rows where both ARG0 and ARG1 are empty are removed.


In [ ]:
# Kept identical to the full notebooks for output parity. The first
# alternative ("0") removes literal zeros, matching the published pipeline.
TRACE_PATTERN = r"0|[*][A-Z]+[*]-\d+|[*][A-Z]+[*]|[*][?][*]|[*]-\d+|[*]"

def remove_traces(text):
    """Strip trace anchors and normalize whitespace."""
    cleaned = re.sub(TRACE_PATTERN, "", text).strip()
    return re.sub(r"\s+", " ", cleaned)

df = pd.DataFrame({
    "SENTENCE": [remove_traces(s) for s in treebanked_sentences],
    "PREDICATES": [remove_traces(s) for s in spans["REL"]],
    "ARG0": [remove_traces(s) for s in spans["ARG0"]],
    "ARG1": [remove_traces(s) for s in spans["ARG1"]],
})
df["Merged_Arguments"] = df["ARG0"] + "|" + df["ARG1"]
df = df[df["Merged_Arguments"] != "|"].reset_index(drop=True)

print(f"Final dataset: {len(df)} rows")
df.head()

## 8. Save and verify the output

The generated CSV is written to `outputs/` and compared with the reference file in `sample_data/`. The assertion fails if any value or row order changes.

For ORL, the final roles map as **ARG0 → Holder**, **REL → Expression**, and **ARG1 → Target**.


In [ ]:
output_dir = DATA_DIR.parent / "outputs"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "sample_output.csv"
df.to_csv(output_path, index=False)

expected_path = DATA_DIR / "expected_output.csv"
expected = pd.read_csv(expected_path, dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(df, expected, check_dtype=False)
print(f"OK: {output_path} matches {expected_path}")

## Next steps

- Run this notebook as-is to verify the pipeline on the sample data.
- For the full WSJ extraction, see `notebooks/[1st_NoteBook]_SRL_DATA_EXTRACTION.ipynb` and `notebooks/[2nd_NoteBook]_SRL_DATA_EXTRACTION.ipynb`. Those notebooks require OntoNotes 5.0 (LDC2013T19) and the PropBank release.
- See `sample_data/README.md` for the file formats and license notes.
